# 🔬 Agricultural Pest Image Retrieval System (CBIR) on Kaggle
Notebook này hướng dẫn chi tiết cách thiết lập môi trường, chạy hệ thống truy vấn ảnh sâu bệnh nông nghiệp 2 giai đoạn (**Automatic Detection & Crop -> CLIP Search**) trên tập dữ liệu **IP102** cho cả **4 Tasks** nhằm thu thập chỉ số Recall@1/5/10 cho báo cáo đánh giá.

### ⚠️ Yêu cầu trước khi chạy:
1. Chắc chắn rằng bạn đã kích hoạt **GPU T4 x2** hoặc **GPU P100** trong phần settings của Kaggle (*Accelerator -> GPU*).
2. Bật kết nối internet cho notebook (*Internet on*).

## 🛠️ Bước 1: Thiết lập môi trường và tải repository

In [1]:
# 1. Clone repository chứa code mới nhất
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# 2. Tải submodule mmyolo
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

-> Đang clone repository từ GitHub...
Cloning into '/kaggle/working/OW_OVD'...
remote: Enumerating objects: 1181, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 1181 (delta 149), reused 113 (delta 50), pack-reused 960 (from 1)
Receiving objects: 100% (1181/1181), 2.11 MiB | 11.07 MiB/s, done.
Resolving deltas: 100% (790/790), done.
/kaggle/working/OW_OVD
-> Đang tải submodule mmyolo...
Cloning into 'third_party/mmyolo'...
remote: Enumerating objects: 4968, done.
remote: Counting objects: 100% (1355/1355), done.
remote: Compressing objects: 100% (290/290), done.
remote: Total 4968 (delta 1148), reused 1065 (delta 1065), pack-reused 3613 (from 1)
Receiving objects: 100% (4968/4968), 3.61 MiB | 20.98 MiB/s, done.
Resolving deltas: 100% (3217/3217), done.


## 📦 Bước 2: Cài đặt Dependencies & Vá lỗi phiên bản MMCV

In [2]:
print("-> 1. Cài đặt các gói PyTorch & Torchvision tương thích cu121...")
!pip install -q torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ pre-built index...")
!pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ...")
!pip install -q matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print("\n-> 4. Cài đặt MMDetection...")
!pip install -q "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install -q --no-build-isolation --no-deps third_party/mmyolo

print("\n-> 6. Vá lỗi kiểm tra phiên bản MMCV...")
import site
import glob

site_dirs = site.getsitepackages()
for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        init_file = os.path.join(s_dir, pkg, "__init__.py")
        if os.path.exists(init_file):
            with open(init_file, 'r', encoding='utf-8') as f:
                content = f.read()
            content = content.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
            content = content.replace("mmcv_maximum_version = '2.1.0'", "mmcv_maximum_version = '2.3.0'")
            with open(init_file, 'w', encoding='utf-8') as f:
                f.write(content)

print("====== Khởi tạo môi trường thành công! ======")

-> 1. Cài đặt các gói PyTorch & Torchvision tương thích cu121...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 96.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 49.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 39.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 87.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 33.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 14.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17

## 🗂️ Bước 3: Tìm kiếm Dataset & Định nghĩa Checkpoints
Định nghĩa thư mục chứa các file checkpoint (`best_coco_Current class AP50_epoch_5.pth`, `t2_best.pth`, `t3_best.pth`, `t4_best.pth`) trên Kaggle.

In [3]:
import glob

# Tự động định vị thư mục dataset IP102 trên Kaggle
dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break

if dataset_root is None:
    paths = glob.glob('/kaggle/input/datasets/nta212/ip102-for-object-detection/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

print(f"-> Thư mục Dataset IP102: {dataset_root}")

# ĐƯỜNG DẪN CHECKPOINT CỦA BẠN TRÊN KAGGLE
CHECKPOINT_DIR = "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3"

# Xác nhận sự tồn tại của file checkpoint mẫu của Task 1 để kiểm tra
t1_ckpt = os.path.join(CHECKPOINT_DIR, "best_coco_Current class AP50_epoch_5.pth")
if os.path.exists(t1_ckpt):
    print(f"-> Phát hiện checkpoint Task 1 tại: {t1_ckpt}")
else:
    print(f"⚠️ Cảnh báo: Chưa tìm thấy checkpoint tại '{t1_ckpt}'. Vui lòng cập nhật biến CHECKPOINT_DIR.")

-> Thư mục Dataset IP102: /kaggle/input/datasets/nta212/ip102-for-object-detection
-> Phát hiện checkpoint Task 1 tại: /kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/best_coco_Current class AP50_epoch_5.pth


## 🛠️ Bước 3b: Sinh các tệp đặc trưng & thuộc tính phụ trợ (Auxiliary Embeddings)
Mô hình YOLO-World/OW-OVD yêu cầu các file vector đặc trưng lớp gán nhãn, thuộc tính, và phân phối mẫu để khởi tạo Box Head. Cell này tự động sinh các file này trong thư mục `data/IP102` trước khi nạp mô hình.

In [4]:
import json
import torch
import numpy as np
import os
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# 1. Khởi tạo các thư mục con
os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# 2. Tải pretrain weights gốc của YOLO-World làm nền tảng nếu cần
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tải pretrained weights gốc...")
    !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
else:
    print("-> Pretrained weights gốc đã có sẵn.")

# 3. Đọc tên các class từ file annotations IP102
ann_path = os.path.join(dataset_root, 'train.json')
if os.path.exists(ann_path):
    with open(ann_path, 'r') as f:
        coco_data = json.load(f)
    categories = sorted(coco_data['categories'], key=lambda x: x['id'])
    class_names = [cat['name'] for cat in categories]
else:
    print("-> Không tìm thấy train.json. Sử dụng danh sách class fallback...")
    from demo_app import IP102_CLASSES
    class_names = IP102_CLASSES[:102]

num_classes = len(class_names)
print(f"-> Tổng số lớp học (classes): {num_classes}")

# 4. Lưu file class_texts.json
class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

# 5. Sinh class embeddings bằng CLIP
print("-> Đang trích xuất text embeddings bằng CLIP...")
model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(model_name)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))
print("-> Đã lưu ip102_gt_embeddings.npy")

# 6. Sinh file task_att_1_embeddings.pth
num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')
print("-> Đã lưu task_att_1_embeddings.pth")

# 7. Sinh file mowod_distribution_sim1.pth
thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("====== Sinh toàn bộ các file đặc trưng phụ trợ thành công! ======")

-> Đang tải pretrained weights gốc...
--2026-08-12 09:40:37--  https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
Resolving huggingface.co (huggingface.co)... 3.171.171.6, 3.171.171.128, 3.171.171.104, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.6|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65bb7a71626a4c209906adf5/09dafb73b0d19d270cf20f7eeac6a7861303a753332d5df9917772ba23e4a47d?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%3B+filename%3D%22yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%22%3B&X-Xet-Cas-Uid=public&Expires=1786531237&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjViYjdhNzE2MjZhNGMyMDk5MDZhZGY1LzA5ZGFmYjczYjBkMTlkMjcwY2YyMGY3ZWVhYzZhNzg2MTMwM2E3NTMzMzJkNWRmOTkxNzc3MmJhMjNlNGE0N2RcXD91c

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.la

-> Đã lưu ip102_gt_embeddings.npy
-> Đã lưu task_att_1_embeddings.pth
====== Sinh toàn bộ các file đặc trưng phụ trợ thành công! ======


## 🔍 Bước 4: Chạy thử tìm kiếm tương đồng trên một ảnh (Query Single Image)
Cell này thực hiện quy trình tìm kiếm ảnh sâu bệnh tương đồng trên một bức ảnh truy vấn bất kỳ và hiển thị kết quả so sánh trực quan.

In [5]:
from IPython.display import Image, display
import glob

# Cấu hình các tham số chạy truy vấn thử nghiệm cho Task 1
t1_config = "configs/open_world/mowod/custom/ip102_t1.py"
t1_checkpoint = os.path.join(CHECKPOINT_DIR, "best_coco_Current class AP50_epoch_5.pth")
gallery_folder = os.path.join(dataset_root, "test")
ann_file = os.path.join(dataset_root, "test.json")

# Tự động tìm kiếm ảnh thực tế bất kỳ trong tập test để truy vấn tránh FileNotFoundError
test_images = glob.glob(os.path.join(dataset_root, "**/*.jpg"), recursive=True)
test_images = [f for f in test_images if "test" in f.replace("\\", "/")]
if test_images:
    query_image_path = test_images[0]
    print(f"-> Tự động tìm thấy ảnh query thực tế: {query_image_path}")
else:
    query_image_path = os.path.join(dataset_root, "test", "00001.jpg")
    print(f"-> Cảnh báo: Không tìm thấy ảnh .jpg. Dùng fallback: {query_image_path}")

# 1. Xây dựng index cho Gallery
print("-> Đang khởi tạo CSDL đặc trưng ảnh mẫu (Gallery Index)... (Có thể mất 2-3 phút)")
!python build_gallery_index.py \
    --config "{t1_config}" \
    --checkpoint "{t1_checkpoint}" \
    --gallery-dir "{gallery_folder}" \
    --ann-file "{ann_file}" \
    --output gallery_index_task1.pkl

# 2. Tiến hành truy vấn ảnh
print("\n-> Đang chạy truy vấn ảnh sâu bệnh...")
!python retrieve.py \
    --config "{t1_config}" \
    --checkpoint "{t1_checkpoint}" \
    --gallery-index gallery_index_task1.pkl \
    --query-image "{query_image_path}" \
    --top-k 5 \
    --anomaly-thr 0.55 \
    --output query_result.jpg

# 3. Hiển thị kết quả tìm kiếm trực quan trực tiếp trong notebook
if os.path.exists("query_result.jpg"):
    display(Image(filename="query_result.jpg"))
else:
    print("Error: Không tìm thấy ảnh kết quả 'query_result.jpg'.")

-> Cảnh báo: Không tìm thấy ảnh .jpg. Dùng fallback: /kaggle/input/datasets/nta212/ip102-for-object-detection/test/00001.jpg
-> Đang khởi tạo CSDL đặc trưng ảnh mẫu (Gallery Index)... (Có thể mất 2-3 phút)
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
      IP102 CBIR GALLERY INDEX BUILDER (OFFLINE)      
-> Loaded annotations mapping for 2713 images.
-> WARNING: Specified directory empty. Scanning parent /kaggle/input/datasets/nta212/ip102-for-object-detection recursively...
-> Filtering gallery images using split annotations: /kaggle/input/datasets/nta212/ip102-for-object-detection/test.json
-> Selected 2713 images belonging to split defined in /kaggle/input/datasets/nta212/ip102-for-object-detection/test.json
-> Total ima

## 📈 Bước 5: Chạy đánh giá đo chỉ số Recall@1/5/10 cho cả 4 Tasks
Cell này chạy đánh giá tuần tự cho cả 4 task và xuất báo cáo điểm số chi tiết từng loài sâu bệnh ra các file markdown.

In [6]:
tasks = [
    {"id": 1, "config": "configs/test/ip102_t1.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "best_coco_Current class AP50_epoch_5.pth")},
    {"id": 2, "config": "configs/test/ip102_t2.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "ip102_t2.pth")},
    {"id": 3, "config": "configs/test/ip102_t3.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "ip102_t3.pth")},
    {"id": 4, "config": "configs/test/ip102_t4.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "ip102_t4.pth")}
,]

for t in tasks:
    task_id = t["id"]
    config_path = t["config"]
    ckpt_path = t["checkpoint"]
    
    print("\n" + "="*60)
    print(f"   ĐANG CHẠY ĐÁNH GIÁ THỰC NGHIỆM CHO TASK {task_id}   ")
    print("="*60)
    
    if not os.path.exists(ckpt_path):
        print(f" Bỏ qua Task {task_id} vì không tìm thấy file checkpoint tại: {ckpt_path}")
        continue
        
    # Khởi chạy script đánh giá
    !python configs/test/evaluate_retrieval_25.py \
        --config "{config_path}" \
        --checkpoint "{ckpt_path}" \
        --dataset-root "{dataset_root}" \
        --query-split val \
        --gallery-split test \
        --query-cache "query_cache_task{task_id}.pkl" \
        --gallery-cache "gallery_cache_task{task_id}.pkl" \
        --output-report "report_task{task_id}.md"


   ĐANG CHẠY ĐÁNH GIÁ THỰC NGHIỆM CHO TASK 1   
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
      IP102 RETRIEVAL METRICS EVALUATION PIPELINE      
-> Reading annotations...
-> WARNING: Standard subfolders empty. Scanning /kaggle/input/datasets/nta212/ip102-for-object-detection recursively to locate image files...
-> Auto-resolved image directory to: /kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/VOC2007/JPEGImages
-> WARNING: Standard subfolders empty. Scanning /kaggle/input/datasets/nta212/ip102-for-object-detection recursively to locate image files...
-> Auto-resolved image directory to: /kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/VOC2007/JPEGImages
-> Found 2176 query images and 

## 📄 Bước 6: Đọc kết quả các Task phục vụ báo cáo

In [7]:
from IPython.display import Markdown, display

for i in [1, 2, 3, 4]:
    report_file = f"report_task{i}.md"
    if os.path.exists(report_file):
        print(f"\n\n KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK {i} (Được đọc từ {report_file}):")
        display(Markdown(filename=report_file))
    else:
        print(f"-> Không tìm thấy báo cáo kết quả của Task {i} (Chưa chạy đánh giá hoặc lỗi file).")



 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK 1 (Được đọc từ report_task1.md):


# Agricultural Pest Retrieval System Evaluation Report

- **Query Split:** `val`
- **Gallery Split:** `test`

## Summary Metrics

| Metric | Macro Average | Weighted Average |
| :--- | :---: | :---: |
| **Recall@1** | 0.5475 | - |
| **Recall@5** | 0.7966 | - |
| **Recall@10** | 0.8859 | - |

## Class-Wise Retrieval Metrics

| Class Name | Query Count | Recall@1 | Recall@5 | Recall@10 |
| :--- | :---: | :---: | :---: | :---: |
| 101 | 470 | 0.8766 | 0.9851 | 0.9936 |
| 14 | 70 | 0.8429 | 0.9286 | 0.9714 |
| 15 | 139 | 0.9856 | 0.9928 | 0.9928 |
| 16 | 68 | 0.5441 | 0.8088 | 0.8529 |
| 18 | 48 | 0.5417 | 0.8333 | 0.8958 |
| 22 | 68 | 0.5147 | 0.7647 | 0.8676 |
| 23 | 33 | 0.4545 | 0.6970 | 0.7879 |
| 24 | 141 | 0.7092 | 0.9220 | 0.9787 |
| 25 | 33 | 0.6970 | 0.7879 | 0.9091 |
| 26 | 39 | 0.5641 | 0.8974 | 0.9487 |
| 37 | 56 | 0.6786 | 0.9286 | 0.9821 |
| 38 | 40 | 0.1500 | 0.4000 | 0.6000 |
| 39 | 66 | 0.2576 | 0.6515 | 0.8333 |
| 45 | 66 | 0.2727 | 0.5758 | 0.8333 |
| 46 | 37 | 0.2162 | 0.4865 | 0.6757 |
| 47 | 56 | 0.3571 | 0.6607 | 0.8214 |
| 48 | 86 | 0.4884 | 0.7558 | 0.8605 |
| 49 | 39 | 0.1795 | 0.5385 | 0.6923 |
| 50 | 70 | 0.6429 | 0.9286 | 0.9714 |
| 51 | 150 | 0.5333 | 0.8800 | 0.9600 |
| 66 | 45 | 0.8222 | 1.0000 | 1.0000 |
| 67 | 54 | 0.9074 | 0.9444 | 0.9444 |
| 69 | 33 | 0.5455 | 0.8182 | 0.8788 |
| 70 | 203 | 0.5567 | 0.9409 | 0.9852 |
| 86 | 66 | 0.3485 | 0.7879 | 0.9091 |




 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK 2 (Được đọc từ report_task2.md):


# Agricultural Pest Retrieval System Evaluation Report

- **Query Split:** `val`
- **Gallery Split:** `test`

## Summary Metrics

| Metric | Macro Average | Weighted Average |
| :--- | :---: | :---: |
| **Recall@1** | 0.5459 | - |
| **Recall@5** | 0.7899 | - |
| **Recall@10** | 0.8663 | - |

## Class-Wise Retrieval Metrics

| Class Name | Query Count | Recall@1 | Recall@5 | Recall@10 |
| :--- | :---: | :---: | :---: | :---: |
| 101 | 470 | 0.8511 | 0.9851 | 0.9936 |
| 14 | 70 | 0.8857 | 0.9429 | 0.9571 |
| 15 | 139 | 0.9640 | 0.9928 | 0.9928 |
| 16 | 68 | 0.6029 | 0.8382 | 0.8824 |
| 18 | 48 | 0.6042 | 0.8333 | 0.9167 |
| 22 | 68 | 0.5441 | 0.7794 | 0.8676 |
| 23 | 33 | 0.3636 | 0.5758 | 0.7576 |
| 24 | 141 | 0.6667 | 0.9362 | 0.9645 |
| 25 | 33 | 0.6667 | 0.8485 | 0.9091 |
| 26 | 39 | 0.5385 | 0.8462 | 0.8974 |
| 37 | 56 | 0.7143 | 0.9464 | 0.9464 |
| 38 | 40 | 0.1500 | 0.3000 | 0.5250 |
| 39 | 66 | 0.2576 | 0.6364 | 0.8333 |
| 45 | 66 | 0.2879 | 0.5606 | 0.8030 |
| 46 | 37 | 0.2162 | 0.4324 | 0.5135 |
| 47 | 56 | 0.3214 | 0.6607 | 0.7679 |
| 48 | 86 | 0.5000 | 0.7674 | 0.8140 |
| 49 | 39 | 0.1282 | 0.5128 | 0.7179 |
| 50 | 70 | 0.5714 | 0.9286 | 0.9429 |
| 51 | 150 | 0.4800 | 0.8867 | 0.9667 |
| 66 | 45 | 0.8444 | 0.9778 | 1.0000 |
| 67 | 54 | 0.9074 | 0.9259 | 0.9444 |
| 69 | 33 | 0.6667 | 0.8788 | 0.8788 |
| 70 | 203 | 0.5813 | 0.9507 | 0.9852 |
| 86 | 66 | 0.3333 | 0.8030 | 0.8788 |




 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK 3 (Được đọc từ report_task3.md):


# Agricultural Pest Retrieval System Evaluation Report

- **Query Split:** `val`
- **Gallery Split:** `test`

## Summary Metrics

| Metric | Macro Average | Weighted Average |
| :--- | :---: | :---: |
| **Recall@1** | 0.5487 | - |
| **Recall@5** | 0.7983 | - |
| **Recall@10** | 0.8698 | - |

## Open-World Anomaly Detection Metrics (HAUF Safeguard)

| Metric | Score |
| :--- | :---: |
| **AUROC (Area Under ROC)** | 0.4660 |
| **FPR@TPR95** | 0.9508 |

### ROC Curve Plot

![ROC Curve](roc_curve_ip102_t3.png)

## Class-Wise Retrieval Metrics

| Class Name | Query Count | Recall@1 | Recall@5 | Recall@10 |
| :--- | :---: | :---: | :---: | :---: |
| 101 | 470 | 0.8787 | 0.9745 | 0.9894 |
| 14 | 70 | 0.8714 | 0.9286 | 0.9571 |
| 15 | 139 | 0.9712 | 0.9928 | 0.9928 |
| 16 | 68 | 0.5588 | 0.8088 | 0.8676 |
| 18 | 48 | 0.5417 | 0.8542 | 0.9167 |
| 22 | 68 | 0.4853 | 0.7647 | 0.8382 |
| 23 | 33 | 0.4242 | 0.6970 | 0.7576 |
| 24 | 141 | 0.7801 | 0.9291 | 0.9716 |
| 25 | 33 | 0.6364 | 0.9091 | 0.9091 |
| 26 | 39 | 0.6410 | 0.8718 | 0.9231 |
| 37 | 56 | 0.7321 | 0.9286 | 0.9643 |
| 38 | 40 | 0.1500 | 0.3250 | 0.5000 |
| 39 | 66 | 0.3182 | 0.7424 | 0.8636 |
| 45 | 66 | 0.2576 | 0.6515 | 0.8485 |
| 46 | 37 | 0.2162 | 0.4595 | 0.5676 |
| 47 | 56 | 0.3036 | 0.6607 | 0.7143 |
| 48 | 86 | 0.4302 | 0.7907 | 0.8721 |
| 49 | 39 | 0.1538 | 0.4103 | 0.7179 |
| 50 | 70 | 0.6429 | 0.9000 | 0.9571 |
| 51 | 150 | 0.5600 | 0.9000 | 0.9467 |
| 66 | 45 | 0.8222 | 1.0000 | 1.0000 |
| 67 | 54 | 0.9074 | 0.9259 | 0.9259 |
| 69 | 33 | 0.5152 | 0.8182 | 0.8485 |
| 70 | 203 | 0.6010 | 0.9409 | 0.9852 |
| 86 | 66 | 0.3182 | 0.7727 | 0.9091 |




 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK 4 (Được đọc từ report_task4.md):


# Agricultural Pest Retrieval System Evaluation Report

- **Query Split:** `val`
- **Gallery Split:** `test`

## Summary Metrics

| Metric | Macro Average | Weighted Average |
| :--- | :---: | :---: |
| **Recall@1** | 0.5457 | - |
| **Recall@5** | 0.8062 | - |
| **Recall@10** | 0.8979 | - |

## Open-World Anomaly Detection Metrics (HAUF Safeguard)

| Metric | Score |
| :--- | :---: |
| **AUROC (Area Under ROC)** | 0.4673 |
| **FPR@TPR95** | 0.9583 |

### ROC Curve Plot

![ROC Curve](roc_curve_ip102_t4.png)

## Class-Wise Retrieval Metrics

| Class Name | Query Count | Recall@1 | Recall@5 | Recall@10 |
| :--- | :---: | :---: | :---: | :---: |
| 101 | 470 | 0.8809 | 0.9830 | 0.9957 |
| 14 | 70 | 0.8429 | 0.9571 | 0.9714 |
| 15 | 139 | 0.9784 | 0.9928 | 0.9928 |
| 16 | 68 | 0.5588 | 0.8088 | 0.9118 |
| 18 | 48 | 0.6250 | 0.8958 | 0.9375 |
| 22 | 68 | 0.5441 | 0.8824 | 0.9412 |
| 23 | 33 | 0.3333 | 0.5455 | 0.7576 |
| 24 | 141 | 0.7234 | 0.9220 | 0.9716 |
| 25 | 33 | 0.5758 | 0.8788 | 0.9394 |
| 26 | 39 | 0.6410 | 0.8718 | 0.9487 |
| 37 | 56 | 0.6964 | 0.9464 | 0.9821 |
| 38 | 40 | 0.1250 | 0.4000 | 0.6000 |
| 39 | 66 | 0.2727 | 0.7121 | 0.8333 |
| 45 | 66 | 0.2576 | 0.5758 | 0.8788 |
| 46 | 37 | 0.1622 | 0.4324 | 0.6757 |
| 47 | 56 | 0.3393 | 0.5714 | 0.8036 |
| 48 | 86 | 0.4767 | 0.7442 | 0.8372 |
| 49 | 39 | 0.2564 | 0.6923 | 0.8205 |
| 50 | 70 | 0.6000 | 0.9143 | 0.9714 |
| 51 | 150 | 0.5600 | 0.8667 | 0.9467 |
| 66 | 45 | 0.8222 | 0.9778 | 1.0000 |
| 67 | 54 | 0.8704 | 0.9259 | 0.9259 |
| 69 | 33 | 0.5152 | 0.8788 | 0.9091 |
| 70 | 203 | 0.6207 | 0.9458 | 0.9704 |
| 86 | 66 | 0.3636 | 0.8333 | 0.9242 |
